In [83]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

In [84]:
# Load a specific sheet
results = pd.read_excel('Concept testing.xlsx', sheet_name='Python2')

overall_results = results[results['Delay'] == 'Overall']

minutes = [3,4,5,6,7,8,9,10,11,12,13,14,15,20,25]
minute_results = {}

for m in minutes:
    minute_results[m] = results[results['Delay'] == f'{m} minute']

In [85]:
# Choose default positions
default_pos = 'top right'
alternate_pos = ['bottom center', 'middle left', 'middle right']

def compute_label_positions(x, y, default_pos='top right', threshold=0.01):
    """
    Assign label positions to avoid overlaps.
    x, y: coordinates of points
    default_pos: the standard position
    threshold: distance below which we consider labels overlapping
    """
    n = len(x)
    positions = [default_pos] * n

    for i in range(n):
        for j in range(i):
            dx = abs(x[i] - x[j])
            dy = abs(y[i] - y[j])
            if dx < threshold or dy < threshold:
                # Overlap detected → switch to alternate position
                positions[i] = alternate_pos[(i + j) % len(alternate_pos)]
    return positions

In [86]:
# for m in minutes: 

#     df = minute_results[m].copy()

#     df['Cat1'] = df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
#     df['Cat2'] = df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)


#     def pareto_front_max(df, x_col, y_col):
#         points = df[[x_col, y_col]].values
#         is_pareto = np.ones(points.shape[0], dtype=bool)

#         for i, point in enumerate(points):
#             if is_pareto[i]:
#                 dominated = np.any(
#                     (points[:, 0] >= point[0]) &
#                     (points[:, 1] >= point[1]) &
#                     ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
#                 )
#                 if dominated:
#                     is_pareto[i] = False

#         return df[is_pareto]


#     pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')


#     fig = go.Figure()

#     positions = compute_label_positions(df['Cat1'].values, df['Cat2'].values)

#     # All designs
#     fig.add_trace(go.Scatter(
#         x=df['Cat1'],
#         y=df['Cat2'],
#         mode='markers+text',
#         text=df['Unnamed: 0'],
#         textposition=positions,
#         cliponaxis=False,
#         name=f'Measures ({m} min)',
#         marker=dict(size=10, color='skyblue', opacity=0.7, line=dict(width=1, color='black')),
#         hovertemplate=(
#             "<b>%{text}</b><br>"
#             "Good for ICE: %{x:.2f}<br>"
#             "Good for Domestic Services: %{y:.2f}"
#             "<extra></extra>"
#         )
#     ))

#     # Pareto front (smooth + circular)
#     fig.add_trace(go.Scatter(
#         x=pareto_df['Cat1'],
#         y=pareto_df['Cat2'],
#         mode='lines+markers',
#         name='Pareto Front',
        
#         line=dict(
#             color='red',
#             width=3,
#             smoothing=1.3
#         ),
#         marker=dict(
#             symbol='circle',
#             size=14,
#             color='red',
#             line=dict(width=2, color='darkred')
#         ),
#         hoverinfo='skip'
#     ))


#     fig.update_layout(
#         title=dict(
#             text='Pareto Front Analysis – {m} Minute Delay',
#             x=0.5,
#             font=dict(size=16)
#         ),
#         xaxis=dict(
#             title='ICE Delay Reduction',
#             gridcolor='rgba(0,0,0,0.15)'
#         ),
#         yaxis=dict(
#             title='Domestic Services Delay Reduction',
#             gridcolor='rgba(0,0,0,0.15)'
#         ),
#         template='plotly_white',
#         legend=dict(
#             x=0.02,
#             y=0.98,
#             bgcolor='rgba(255,255,255,0.8)'
#         ),
#         width=900,
#         height=650
#     )

#     fig.show()

In [87]:
# Pareto front function (define ONCE)
def pareto_front_max(df, x_col, y_col):
    points = df[[x_col, y_col]].values
    is_pareto = np.ones(points.shape[0], dtype=bool)

    for i, point in enumerate(points):
        if is_pareto[i]:
            dominated = np.any(
                (points[:, 0] >= point[0]) &
                (points[:, 1] >= point[1]) &
                ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
            )
            if dominated:
                is_pareto[i] = False

    return df[is_pareto]

#plot the overall graph

df = overall_results.copy()

# Compute category scores
df['Cat1'] = df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
df['Cat2'] = df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)

pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')

fig = go.Figure()

positions = compute_label_positions(df['Cat1'].values, df['Cat2'].values)

# All designs
fig.add_trace(go.Scatter(
    x=df['Cat1'],
    y=df['Cat2'],
    mode='markers+text',
    text=df['Unnamed: 0'],
    textposition=positions,
    cliponaxis=False,
    name=f'Measures',
    marker=dict(size=10, color='skyblue', opacity=0.7, line=dict(width=1, color='black')),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Good for ICE: %{x:.2f}<br>"
        "Good for Domestic Services: %{y:.2f}"
        "<extra></extra>"
    )
))


# Pareto front
fig.add_trace(go.Scatter(
    x=pareto_df['Cat1'],
    y=pareto_df['Cat2'],
    mode='lines+markers',
    name=f'Pareto Front',
    line=dict(
        color='red',
        width=3,
        smoothing=1.3
    ),
    marker=dict(
        symbol='circle',
        size=14,
        color='red',
        line=dict(width=2, color='darkred')
    ),
    hoverinfo='skip'
))

# Layout
fig.update_layout(
    title=dict(
        text=f'Pareto Front Analysis',
        x=0.5,
        font=dict(size=16)
    ),
    xaxis_title='ICE Delay Reduction',
    yaxis_title='Domestic Services Delay Reduction',
    template='plotly_white',
    width=900,
    height=650
)

fig.write_image(f"Pareto_graphs/overall_pareto.png")

# --------------------------------------------------
# Loop over minutes
# --------------------------------------------------
for m in minutes:

    df = minute_results[m].copy()

    # Compute category scores
    df['Cat1'] = df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
    df['Cat2'] = df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)

    pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')

    fig = go.Figure()

    # All designs
    positions = compute_label_positions(df['Cat1'].values, df['Cat2'].values)

    # All designs
    fig.add_trace(go.Scatter(
        x=df['Cat1'],
        y=df['Cat2'],
        mode='markers+text',
        text=df['Unnamed: 0'],
        textposition=positions,
        cliponaxis=False,
        name=f'Measures ({m} min)',
        marker=dict(size=10, color='skyblue', opacity=0.7, line=dict(width=1, color='black')),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Good for ICE: %{x:.2f}<br>"
            "Good for Domestic Services: %{y:.2f}"
            "<extra></extra>"
        )
    ))


    # Pareto front
    fig.add_trace(go.Scatter(
        x=pareto_df['Cat1'],
        y=pareto_df['Cat2'],
        mode='lines+markers',
        name=f'Pareto Front ({m} min)',
        line=dict(
            color='red',
            width=3,
            smoothing=1.3
        ),
        marker=dict(
            symbol='circle',
            size=14,
            color='red',
            line=dict(width=2, color='darkred')
        ),
        hoverinfo='skip'
    ))

    # Layout
    fig.update_layout(
        title=dict(
            text=f'Pareto Front Analysis – {m} Minute Delay',
            x=0.5,
            font=dict(size=16)
        ),
        xaxis_title='ICE Delay Reduction',
        yaxis_title='Domestic Services Delay Reduction',
        template='plotly_white',
        width=900,
        height=650
    )

    fig.write_image(f"Pareto_graphs/pareto_{m}_minute.png")
    #fig.show()


In [88]:
import numpy as np
import plotly.graph_objects as go

# --------------------------------------------------
# Pareto front function
# --------------------------------------------------
def pareto_front_max(df, x_col, y_col):
    points = df[[x_col, y_col]].values
    is_pareto = np.ones(points.shape[0], dtype=bool)

    for i, point in enumerate(points):
        if is_pareto[i]:
            dominated = np.any(
                (points[:, 0] >= point[0]) &
                (points[:, 1] >= point[1]) &
                ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
            )
            if dominated:
                is_pareto[i] = False

    return df[is_pareto]

# --------------------------------------------------
# Define measure names
# --------------------------------------------------
measure_names = [
    "Cancel the Arriva stop train 31200 in Hengelo in case of ICE delay.",
    "Grant the ICE priority over the Arriva stop train 31200 at Oldenzaal in case of ICE delay.",
    "Do not grant the ICE priority over the Arriva stop train 31200 when the ICE is delayed.",
    "Remove Hengelo as a stop for the ICE in case of ICE delay",
    "Use the overtaking track at Rijssen in case of ICE delay.",
    "Allow a delayed ICE to short-turn in Hengelo.",
    "Use predefined alternative timetable paths for the ICE.",
    "Remove Hengelo as a stop for the ICE in the timetable.",
    "Reintroduce stopping at Almelo instead of Hengelo.",
    "Construct an additional platform at Hengelo."
]

# --------------------------------------------------
# Precompute frames for all minutes
# --------------------------------------------------
frames = []

for m in minutes:
    df = minute_results[m].copy()
    
    # Compute Cat1 and Cat2
    df['ICE'] = df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
    df['Domestic'] = df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)

    # Map measure descriptions
    measure_desc_map = dict(zip([f"M{i}" for i in range(1, 11)], measure_names))
    df['MeasureDesc'] = df['Unnamed: 0'].map(measure_desc_map)

    # Compute label positions
    label_positions = compute_label_positions(df['ICE'].values, df['Domestic'].values)

    # Pareto front
    pareto_df = pareto_front_max(df, 'ICE', 'Domestic').sort_values('ICE')

    frames.append(
        go.Frame(
            name=str(m),
            data=[
                # All designs
                go.Scatter(
                    x=df['ICE'],
                    y=df['Domestic'],
                    mode='markers+text',
                    text=df['Unnamed: 0'],  # M1, M2, etc.
                    textposition=label_positions,
                    cliponaxis=False,
                    marker=dict(size=10, color='skyblue', opacity=0.7, line=dict(width=1, color='black')),
                    name='Designs',
                    hovertemplate=(
                        "<b>%{text}</b><br>" +
                        "%{customdata}<br>" +
                        "ICE: %{x:.2f}<br>" +
                        "Domestic: %{y:.2f}<extra></extra>"
                    ),
                    customdata=df['MeasureDesc']
                ),
                # Pareto front
                go.Scatter(
                    x=pareto_df['ICE'],
                    y=pareto_df['Domestic'],
                    mode='lines+markers',
                    line=dict(color='red', width=3, smoothing=1.3),
                    marker=dict(symbol='circle', size=14, color='red', line=dict(width=2, color='darkred')),
                    name='Pareto Front',
                    text=pareto_df['Unnamed: 0'],
                    hovertemplate=(
                        "<b>%{text}</b><br>" +
                        "%{customdata}<br>" +
                        "ICE: %{x:.2f}<br>" +
                        "Domestic: %{y:.2f}<extra></extra>"
                    ),
                    customdata=pareto_df['MeasureDesc']
                )
            ],
            layout=go.Layout(title_text=f'Pareto Front Analysis – {m} Minute Delay')
        )
    )

# --------------------------------------------------
# Initial figure (first minute)
# --------------------------------------------------
init_df = minute_results[minutes[0]].copy()
init_df['ICE'] = init_df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
init_df['Domestic'] = init_df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)

measure_desc_map = dict(zip([f"M{i}" for i in range(1, 11)], measure_names))
init_df['MeasureDesc'] = init_df['Unnamed: 0'].map(measure_desc_map)

init_positions = compute_label_positions(init_df['ICE'].values, init_df['Domestic'].values)
init_pareto = pareto_front_max(init_df, 'ICE', 'Domestic').sort_values('ICE')

fig = go.Figure(
    data=[
        go.Scatter(
            x=init_df['ICE'],
            y=init_df['Domestic'],
            mode='markers+text',
            text=init_df['Unnamed: 0'],  # Only M1–M10
            textposition=init_positions,
            cliponaxis=False,
            name='Designs',
            marker=dict(size=10, color='skyblue', opacity=0.7, line=dict(width=1, color='black')),
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "%{customdata}<br>" +
                "ICE: %{x:.2f}<br>" +
                "Domestic: %{y:.2f}<extra></extra>"
            ),
            customdata=init_df['MeasureDesc']
        ),
        go.Scatter(
            x=init_pareto['ICE'],
            y=init_pareto['Domestic'],
            mode='lines+markers',
            name='Pareto Front',
            line=dict(color='red', width=3, smoothing=1.3),
            marker=dict(symbol='circle', size=14, color='red', line=dict(width=2, color='darkred')),
            text=init_pareto['Unnamed: 0'],
            hovertemplate=(
                "<b>%{text}</b><br>" +
                "%{customdata}<br>" +
                "ICE: %{x:.2f}<br>" +
                "Domestic: %{y:.2f}<extra></extra>"
            ),
            customdata=init_pareto['MeasureDesc']
        )
    ],
    frames=frames
)

# --------------------------------------------------
# Slider + layout
# --------------------------------------------------
fig.update_layout(
    template='plotly_white',
    width=900,
    height=650,
    xaxis_title='ICE',
    yaxis_title='Domestic',
    title=f'Pareto Front Analysis – {minutes[0]} Minute Delay',
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "Delay: "},
        "pad": {"t": 50},
        "steps": [
            {
                "method": "animate",
                "label": f"{m} min",
                "args": [
                    [str(m)],
                    {"mode": "immediate", "frame": {"duration": 600}, "transition": {"duration": 300}}
                ]
            }
            for m in minutes
        ]
    }]
)

fig.write_html("pareto_slider.html")
